# TagScript

## Objetivo

Construir un **traductor** (compilador fuente a fuente) para un lenguaje propio, **TagScript**, que permita describir una página web con muy poco texto y obtener automáticamente `index.html` y `style.css`.

## Ejemplo

```
BTN#enviar.btn-primary "Enviar"
        ↓ se traduce a
<button id="enviar" class="btn-primary">Enviar</button>
```

## Alcance

**Incluye**

- 20 etiquetas (`TITULO`, `PARRAFO`, `BTN`, `LINK`, `DIV`, `LISTA`, `FORM`, `IMAGEN`, `INPUT`, etc.).
- Selectores de id y clase (`DIV#caja.card`), atributos (`[href="..."]`), texto y bloques anidados `{ }`.
- Variables (`VAR`), interpolación en textos (`"Hola $nombre"`), componentes con parámetros (`COMPONENTE`), contenido insertado con `HIJOS`, y bucles `REPETIR` con índice `$i`.
- Configuración de la página (`PAGINA [titulo, tema, color, idioma, descripcion]`).
- Errores con línea y avisos de accesibilidad.
- CSS generado, con tema claro, oscuro o automático.
- Un agente con Ollama que usa al compilador como verificador.

**No incluye**

- Condicionales, expresiones aritméticas ni JavaScript.
- Estilos personalizados propiedad por propiedad (se usan clases predefinidas).
- Importación de otros archivos ni etiquetas fuera de la tabla del lenguaje.
- El agente necesita Ollama instalado; si no está disponible, la notebook usa un **modo simulado** para poder demostrarlo igual.

## Arquitectura

```
 texto TagScript
      │
      ▼
 ┌──────────┐   tokens   ┌─────────────┐    AST    ┌─────────────┐  árbol resuelto  ┌────────────┐
 │  LÉXICO  │ ─────────► │ SINTÁCTICO  │ ────────► │  SEMÁNTICO  │ ───────────────► │ GENERADOR  │ ─► index.html
 │ lexer.py │            │parser (yacc)│           │ semantico.py│                  │ + estilos  │ ─► style.css
 └──────────┘            └─────────────┘           └─────────────┘                  └────────────┘

 AGENTE (Ollama):  pedido en español ─► LLM ─► código TagScript ─► compilador ──ok──► HTML
                                          ▲                            │
                                          └──── errores del compilador ┘  (hasta N intentos)
```

| Módulo | Fase | Responsabilidad |
|---|---|---|
| `nodos.py` | — | Tabla de etiquetas, clases del AST y diagnósticos |
| `lexer.py` | Léxica | Texto → tokens (`ply.lex`) |
| `tagscript_parser.py` | Sintáctica | Tokens → AST (`ply.yacc`). **No genera HTML** |
| `semantico.py` | Semántica | Resuelve variables, componentes y bucles; valida |
| `generador.py`, `estilos.py` | Generación | Árbol resuelto → HTML y CSS |
| `compilador.py` | — | Une las cuatro fases en una sola función `compilar()` |
| `ver.py` | — | Muestra en pantalla lo que hace cada fase |

**Decisión de diseño:** el parser solo construye el AST. La generación de HTML ocurre después, sobre un árbol ya validado. Así los componentes y las variables se resuelven en un lugar (el análisis semántico) y no mezclados con la gramática.

### Estructuras compartidas (`nodos.py`)

Contiene la **tabla de etiquetas** (`TITULO → h1`, `BTN → button`...), los nodos del **AST** que produce el parser (`Elemento`, `Componente`, `Llamada`, `Repetir`, `Asignacion`, `Pagina`, `Hijos`) y el nodo `Nodo`, que representa un elemento HTML ya resuelto. También define `Diag`, la estructura de los diagnósticos (`nivel`, `linea`, `mensaje`), con nivel `error` o `aviso`.

In [ ]:
%%writefile build_tagscript/nodos.py
"""Estructuras compartidas: tabla de etiquetas, nodos del AST y diagnósticos."""
from dataclasses import dataclass, field
from typing import Any, List, Optional, Tuple
from collections import namedtuple

# Etiqueta TagScript -> (etiqueta HTML, ¿es vacía?)
TAGS = {
    'TITULO':   ('h1', False),
    'SUBTITULO': ('h2', False),
    'ENCABEZADO': ('h3', False),
    'PARRAFO':  ('p', False),
    'TEXTO':    ('span', False),
    'CODIGO':   ('code', False),
    'BTN':      ('button', False),
    'LINK':     ('a', False),
    'DIV':      ('div', False),
    'SECCION':  ('section', False),
    'CABECERA': ('header', False),
    'PIE':      ('footer', False),
    'NAV':      ('nav', False),
    'LISTA':    ('ul', False),
    'ITEM':     ('li', False),
    'FORM':     ('form', False),
    'CAMPO':    ('label', False),
    'IMAGEN':   ('img', True),
    'INPUT':    ('input', True),
    'LINEA':    ('hr', True),
}

Diag = namedtuple('Diag', 'nivel linea mensaje')   # nivel: 'error' | 'aviso'


# ---------- AST (lo que produce el parser) ----------
@dataclass
class Var:
    nombre: str
    linea: int = 0

@dataclass
class Elemento:
    tag: str
    selectores: List[Tuple[str, str]]      # [('id','x'), ('clase','y')]
    attrs: List[Tuple[str, Any]]           # [(nombre, str | Var)]
    texto: Any                             # str | Var | None
    cuerpo: Optional[list]                 # None = sin bloque { }
    linea: int = 0

@dataclass
class Componente:
    nombre: str
    params: List[str]
    cuerpo: list
    linea: int = 0

@dataclass
class Llamada:
    nombre: str
    args: list
    cuerpo: Optional[list]
    linea: int = 0

@dataclass
class Repetir:
    veces: int
    cuerpo: list
    linea: int = 0

@dataclass
class Asignacion:
    nombre: str
    valor: Any
    linea: int = 0

@dataclass
class Pagina:
    attrs: List[Tuple[str, Any]]
    linea: int = 0

@dataclass
class Hijos:
    linea: int = 0


# ---------- Árbol ya resuelto (lo que consume el generador) ----------
@dataclass
class Nodo:
    tag: str
    origen: str
    attrs: List[Tuple[str, str]]
    texto: Optional[str]
    hijos: List['Nodo']
    vacio: bool
    linea: int


@dataclass
class Resultado:
    html: Optional[str]
    css: Optional[str]
    diagnosticos: List[Diag] = field(default_factory=list)

    @property
    def ok(self):
        return self.html is not None

## Aspectos léxicos

El analizador léxico convierte el texto en una secuencia de **tokens**.

| Token | Expresión regular | Ejemplo |
|---|---|---|
| `ETIQUETA` | identificador que está en la tabla de etiquetas | `BTN`, `DIV` |
| `COMPONENTE`, `REPETIR`, `VAR`, `PAGINA`, `HIJOS` | palabras reservadas | `REPETIR` |
| `IDENT` | `[A-Za-z_][\w-]*` | `href`, `tarjeta`, `data-id` |
| `VARIABLE` | `\$[A-Za-z_]\w*` | `$marca` |
| `HASHID` | `\#[A-Za-z_][\w-]*` | `#enviar` |
| `CLASE` | `\.[A-Za-z_][\w-]*` | `.card` |
| `STRING` | `"(?:[^"\\\n]\|\\.)*"` | `"Hola"` |
| `NUMERO` | `\d+` | `3` |
| `CORCHETE_IZQ/DER`, `LLAVE_IZQ/DER`, `PAREN_IZQ/DER`, `IGUAL` | `\[` `\]` `\{` `\}` `\(` `\)` `=` | |

**Se ignoran:** espacios, tabulaciones, `\r`, **comas** (son opcionales) y **comentarios** `// ...` hasta fin de línea.

**Decisiones léxicas**

- **Palabras reservadas:** no tienen una regla propia. Se reconocen con la regla de `IDENT` y se consulta una tabla; si el texto está en ella, el token cambia de tipo. Así `BTNX` es un identificador común y no `BTN` seguido de `X`.
- **Los textos entre comillas son un solo token.** Por eso `"http://sitio.com/#inicio"` no se confunde con un comentario ni con un id.
- **Un texto no puede cruzar líneas:** una comilla sin cerrar se detecta en la línea donde empieza.
- **Errores léxicos:** carácter no reconocido o comilla sin cerrar. Se guardan con su línea y el análisis continúa, para informar varios errores juntos.
- **Contador de líneas:** la regla `t_newline` actualiza `lineno`, y de ahí salen los números de línea de todos los errores posteriores.

In [ ]:
%%writefile build_tagscript/lexer.py
"""Fase 1 - Análisis léxico: convierte el texto en tokens."""
import re
import ply.lex as lex
from nodos import TAGS, Diag

PALABRAS = {'COMPONENTE', 'REPETIR', 'VAR', 'PAGINA', 'HIJOS'}

tokens = (
    'ETIQUETA', 'IDENT', 'VARIABLE', 'HASHID', 'CLASE', 'STRING', 'NUMERO',
    'CORCHETE_IZQ', 'CORCHETE_DER', 'LLAVE_IZQ', 'LLAVE_DER',
    'PAREN_IZQ', 'PAREN_DER', 'IGUAL',
) + tuple(sorted(PALABRAS))

errores_lexicos = []

t_CORCHETE_IZQ = r'\['
t_CORCHETE_DER = r'\]'
t_LLAVE_IZQ = r'\{'
t_LLAVE_DER = r'\}'
t_PAREN_IZQ = r'\('
t_PAREN_DER = r'\)'
t_IGUAL = r'='
t_ignore_COMENTARIO = r'//[^\n]*'
t_ignore = ' \t\r,'          # las comas son opcionales


def t_STRING(t):
    r'"(?:[^"\\\n]|\\.)*"'
    t.value = re.sub(r'\\(["\\])', r'\1', t.value[1:-1])
    return t

def t_NUMERO(t):
    r'\d+'
    t.value = int(t.value)
    return t

def t_VARIABLE(t):
    r'\$[A-Za-z_]\w*'
    return t

def t_HASHID(t):
    r'\#[A-Za-z_][\w-]*'
    return t

def t_CLASE(t):
    r'\.[A-Za-z_][\w-]*'
    return t

def t_IDENT(t):
    r'[A-Za-z_][\w-]*'
    if t.value in TAGS:
        t.type = 'ETIQUETA'
    elif t.value in PALABRAS:
        t.type = t.value
    return t

def t_newline(t):
    r'\n+'
    t.lexer.lineno += len(t.value)

def t_error(t):
    if t.value[0] == '"':
        resto = t.value.find('\n')
        errores_lexicos.append(Diag('error', t.lexer.lineno,
                                    "Error léxico: comilla sin cerrar"))
        t.lexer.skip(resto if resto > 0 else len(t.value))
        return
    errores_lexicos.append(Diag('error', t.lexer.lineno,
                                f"Error léxico: carácter no reconocido '{t.value[0]}'"))
    t.lexer.skip(1)

lexer = lex.lex()

**Demostración léxica:** una línea de TagScript convertida en tokens. Los comentarios y espacios desaparecen.

In [ ]:
import lexer as lx

muestra = 'BTN#enviar.btn-primary [type="submit"] "Enviar"   // un comentario'
print("ENTRADA:", muestra, "\n")

lx.errores_lexicos.clear()
lx.lexer.lineno = 1
lx.lexer.input(muestra)

print(f"{'línea':<7}{'token':<15}valor")
for tok in iter(lx.lexer.token, None):
    print(f"{tok.lineno:<7}{tok.type:<15}{tok.value!r}")

## Aspectos sintácticos

**Gramática** (los símbolos en mayúscula son tokens; `ε` es la producción vacía):

```
programa     ::= lista
lista        ::= lista sentencia | ε
sentencia    ::= elemento | componente | llamada | repetir | asignacion | pagina | hijos

elemento     ::= ETIQUETA selectores atributos texto cuerpo
selectores   ::= selectores selector | ε
selector     ::= HASHID | CLASE
atributos    ::= '[' lista_attr ']' | ε
lista_attr   ::= lista_attr atributo | ε
atributo     ::= IDENT '=' valor
valor        ::= STRING | VARIABLE
texto        ::= valor | ε
cuerpo       ::= '{' lista '}' | ε

componente   ::= COMPONENTE IDENT '(' parametros ')' '{' lista '}'
parametros   ::= parametros VARIABLE | ε
llamada      ::= IDENT '(' argumentos ')' cuerpo
argumentos   ::= argumentos valor | ε
repetir      ::= REPETIR NUMERO '{' lista '}'
asignacion   ::= VAR VARIABLE '=' valor
pagina       ::= PAGINA atributos
hijos        ::= HIJOS
```

**Decisiones de diseño**

- **Recursión por la izquierda** (`lista ::= lista sentencia`): es la forma natural para el análisis ascendente LALR que usa PLY.
- **Partes opcionales con `ε`:** un elemento puede llevar o no selectores, atributos, texto y bloque. Cada una se decide con un solo token de anticipación.
- **`VAR` para declarar variables:** si la asignación empezara directamente con `$x`, la gramática no podría distinguirla de un texto `$x` que pertenece al elemento anterior. La palabra reservada elimina esa ambigüedad.
- **El parser solo construye el AST.** Cada regla `p_*` crea un nodo y no genera HTML.
- **Un error sintáctico se informa con `p_error`**, que lanza una excepción propia con la línea y el token inesperado. Un fin de archivo inesperado sugiere que falta cerrar `}`, `)` o `]`.

In [ ]:
%%writefile build_tagscript/tagscript_parser.py
"""Fase 2 - Análisis sintáctico: tokens -> AST (sin generar HTML aquí)."""
import ply.yacc as yacc
from lexer import tokens, lexer, errores_lexicos
from nodos import (Var, Elemento, Componente, Llamada, Repetir, Asignacion,
                   Pagina, Hijos, Diag)


class ErrorSintactico(Exception):
    def __init__(self, linea, mensaje):
        super().__init__(mensaje)
        self.linea, self.mensaje = linea, mensaje


def p_programa(p):
    'programa : lista'
    p[0] = p[1]

def p_lista(p):
    '''lista : lista sentencia
             | empty'''
    p[0] = p[1] + [p[2]] if len(p) == 3 else []

def p_sentencia(p):
    '''sentencia : elemento
                 | componente
                 | llamada
                 | repetir
                 | asignacion
                 | pagina
                 | hijos'''
    p[0] = p[1]

def p_elemento(p):
    'elemento : ETIQUETA selectores atributos texto cuerpo'
    p[0] = Elemento(p[1], p[2], p[3], p[4], p[5], p.lineno(1))

def p_selectores(p):
    '''selectores : selectores selector
                  | empty'''
    p[0] = p[1] + [p[2]] if len(p) == 3 else []

def p_selector(p):
    '''selector : HASHID
                | CLASE'''
    tipo = 'id' if p.slice[1].type == 'HASHID' else 'clase'
    p[0] = (tipo, p[1][1:])

def p_atributos(p):
    '''atributos : CORCHETE_IZQ lista_attr CORCHETE_DER
                 | empty'''
    p[0] = p[2] if len(p) == 4 else []

def p_lista_attr(p):
    '''lista_attr : lista_attr atributo
                  | empty'''
    p[0] = p[1] + [p[2]] if len(p) == 3 else []

def p_atributo(p):
    'atributo : IDENT IGUAL valor'
    p[0] = (p[1], p[3])

def p_valor(p):
    '''valor : STRING
             | VARIABLE'''
    p[0] = p[1] if p.slice[1].type == 'STRING' else Var(p[1][1:], p.lineno(1))

def p_texto(p):
    '''texto : valor
             | empty'''
    p[0] = p[1]

def p_cuerpo(p):
    '''cuerpo : LLAVE_IZQ lista LLAVE_DER
              | empty'''
    p[0] = p[2] if len(p) == 4 else None

def p_componente(p):
    'componente : COMPONENTE IDENT PAREN_IZQ parametros PAREN_DER LLAVE_IZQ lista LLAVE_DER'
    p[0] = Componente(p[2], p[4], p[7], p.lineno(1))

def p_parametros(p):
    '''parametros : parametros VARIABLE
                  | empty'''
    p[0] = p[1] + [p[2][1:]] if len(p) == 3 else []

def p_llamada(p):
    'llamada : IDENT PAREN_IZQ argumentos PAREN_DER cuerpo'
    p[0] = Llamada(p[1], p[3], p[5], p.lineno(1))

def p_argumentos(p):
    '''argumentos : argumentos valor
                  | empty'''
    p[0] = p[1] + [p[2]] if len(p) == 3 else []

def p_repetir(p):
    'repetir : REPETIR NUMERO LLAVE_IZQ lista LLAVE_DER'
    p[0] = Repetir(p[2], p[4], p.lineno(1))

def p_asignacion(p):
    'asignacion : VAR VARIABLE IGUAL valor'
    p[0] = Asignacion(p[2][1:], p[4], p.lineno(1))

def p_pagina(p):
    'pagina : PAGINA atributos'
    p[0] = Pagina(p[2], p.lineno(1))

def p_hijos(p):
    'hijos : HIJOS'
    p[0] = Hijos(p.lineno(1))

def p_empty(p):
    'empty :'
    p[0] = None

def p_error(p):
    if p is None:
        linea = lexer.lineno
        if str(getattr(lexer, 'lexdata', '')).endswith('\n'):
            linea -= 1                      # no contar el salto de línea final
        raise ErrorSintactico(linea,
            "fin de archivo inesperado (¿falta cerrar un '}', ')' o ']'?)")
    visto = f'"{p.value}"' if p.type == 'STRING' else str(p.value)
    raise ErrorSintactico(p.lineno, f"no se esperaba {visto} en este punto")


parser = yacc.yacc(write_tables=False, debug=False)


def parsear(fuente):
    """Devuelve (ast, diagnosticos). ast es None si hubo errores."""
    errores_lexicos.clear()
    lexer.lineno = 1
    try:
        ast = parser.parse(fuente, lexer=lexer)
    except ErrorSintactico as e:
        diag = list(errores_lexicos) + [Diag('error', e.linea, f"Error sintáctico: {e.mensaje}")]
        return None, diag
    diag = list(errores_lexicos)
    return (None if diag else ast), diag

**Demostración sintáctica:** el AST de un programa pequeño. Todavía aparecen `Componente`, `Repetir` y `$variables` sin resolver. Después, un programa mal formado.

In [ ]:
from pprint import pprint
from tagscript_parser import parsear

programa = '''VAR $marca = "TagScript"
COMPONENTE tarjeta($titulo) {
    DIV.card { ENCABEZADO $titulo }
}
REPETIR 2 { tarjeta("Número $i") }
'''
print("AST de un programa correcto:\n")
ast, diag = parsear(programa)
pprint(ast, width=110)

print("\n" + "-" * 70)
print("Un programa mal formado (falta el valor del atributo):\n")
roto = 'LINK [href] "Ir al sitio"'
ast, diag = parsear(roto)
print("AST:", ast)
for d in diag:
    print(f"{d.nivel} en la línea {d.linea}: {d.mensaje}")

## Aspectos semánticos

El análisis semántico recorre el AST y ejecuta las **acciones semánticas** del lenguaje. Su salida es un árbol de nodos HTML ya resueltos: sin componentes, sin bucles y sin variables.

| Construcción | Acción semántica |
|---|---|
| `VAR $x = "v"` | Guarda `x` en el ámbito actual. Cada bloque `{ }` y cada llamada a un componente trabajan con una copia del ámbito. |
| `$x` como texto o valor de atributo | Se busca en el ámbito. Si no existe: **error semántico**. |
| `"Hola $x"` | **Interpolación:** reemplaza `$x` solo si está definida. |
| `COMPONENTE f($a $b) { ... }` | Lo registra en la tabla de componentes. Se puede usar antes de definirlo; un nombre repetido es un error. |
| `f("1", "2")` | Verifica que exista (con sugerencia por similitud: *"¿Quisiste decir...?"*) y que la cantidad de argumentos coincida. Crea un ámbito con los parámetros y expande el cuerpo. |
| `HIJOS` | Inserta el bloque `{ }` de la llamada, evaluado en el ámbito de quien llama. |
| `REPETIR n { ... }` | Expande el cuerpo `n` veces con `$i = 1..n`. |
| `#id`, `.clase`, `[atributos]` | Se fusionan: un solo id, clases combinadas, y un atributo repetido es error. |
| `IMAGEN "x"` / `INPUT "x"` | El texto pasa a ser `src` / `placeholder`. Estas etiquetas no admiten bloque `{ }`. |
| `PAGINA [...]` | Valida las claves permitidas, el tema (`claro`/`oscuro`/`auto`) y que el color sea hexadecimal. |
| Validación final | **Errores:** id duplicado. **Avisos:** `IMAGEN` sin `alt`, `LINK` sin `href`, `ITEM` fuera de `LISTA`. |
| Protecciones | Recursión de componentes limitada a 25 niveles y máximo de 5000 elementos, para que un programa mal escrito no cuelgue la máquina. |

**Errores y avisos.** Un *error* impide generar la salida. Un *aviso* (por ejemplo, accesibilidad) no la impide, pero se informa.

**Diferencia entre un error sintáctico y uno semántico.** `LINK [href] "Ir"` es **sintáctico**: la forma está mal, porque falta el valor. `PARRAFO $nombre` es **semántico**: la forma es correcta, pero `$nombre` nunca fue definida.

In [ ]:
%%writefile build_tagscript/semantico.py
"""Fase 3 - Análisis semántico: resuelve variables, componentes y bucles.
Entrada: AST del parser. Salida: árbol de Nodo listo para generar HTML."""
import re
import difflib
from nodos import (TAGS, Var, Elemento, Componente, Llamada, Repetir,
                   Asignacion, Pagina, Hijos, Nodo, Diag)

MAX_LLAMADAS = 25      # profundidad máxima de componentes (evita recursión infinita)
MAX_NODOS = 5000       # evita REPETIR gigantes
HEX = re.compile(r'^#(?:[0-9a-fA-F]{3}|[0-9a-fA-F]{6})$')
CLAVES_PAGINA = ('titulo', 'tema', 'color', 'idioma', 'descripcion')
INTERP = re.compile(r'\$([A-Za-z_]\w*)')


class _Abortar(Exception):
    pass


class Analizador:
    def __init__(self):
        self.componentes = {}
        self.diagnosticos = []
        self.config = {}
        self.cuenta = 0

    # ---- utilidades ----
    def error(self, linea, msg):
        self.diagnosticos.append(Diag('error', linea, f"Error semántico: {msg}"))

    def aviso(self, linea, msg):
        self.diagnosticos.append(Diag('aviso', linea, msg))

    def valor(self, v, scope):
        """Resuelve $variables; en textos literales interpola $nombre si existe."""
        if isinstance(v, Var):
            if v.nombre not in scope:
                self.error(v.linea, f"la variable ${v.nombre} no está definida")
                return ""
            return str(scope[v.nombre])
        return INTERP.sub(lambda m: str(scope.get(m.group(1), m.group(0))), v)

    # ---- punto de entrada ----
    def analizar(self, ast):
        for n in ast:                       # permite usar un componente antes de definirlo
            if isinstance(n, Componente):
                self.registrar(n)
        try:
            nodos = self.expandir(ast, {}, None, 0)
        except _Abortar:
            return [], self.config
        self.validar(nodos, set())
        return nodos, self.config

    def registrar(self, c):
        previo = self.componentes.get(c.nombre)
        if previo is not None and previo is not c:
            self.error(c.linea, f"el componente '{c.nombre}' ya fue definido (línea {previo.linea})")
            return
        if len(set(c.params)) != len(c.params):
            self.error(c.linea, f"'{c.nombre}' tiene parámetros repetidos")
        self.componentes[c.nombre] = c

    # ---- expansión ----
    def expandir(self, nodos, scope, ctx, prof):
        out = []
        for n in nodos:
            if isinstance(n, Componente):
                self.registrar(n)
            elif isinstance(n, Asignacion):
                scope[n.nombre] = self.valor(n.valor, scope)
            elif isinstance(n, Pagina):
                self.pagina(n, scope)
            elif isinstance(n, Hijos):
                if ctx is None:
                    self.error(n.linea, "HIJOS solo puede usarse dentro de un COMPONENTE")
                elif ctx[0] is not None:
                    out += self.expandir(ctx[0], dict(ctx[1]), ctx[2], ctx[3])
            elif isinstance(n, Repetir):
                if n.veces > MAX_NODOS:
                    self.error(n.linea, f"REPETIR {n.veces} supera el máximo permitido ({MAX_NODOS})")
                    continue
                for i in range(1, n.veces + 1):
                    sc = dict(scope)
                    sc['i'] = str(i)
                    out += self.expandir(n.cuerpo, sc, ctx, prof)
            elif isinstance(n, Llamada):
                out += self.llamar(n, scope, ctx, prof)
            elif isinstance(n, Elemento):
                out.append(self.elemento(n, scope, ctx, prof))
        return out

    def llamar(self, n, scope, ctx, prof):
        comp = self.componentes.get(n.nombre)
        if comp is None:
            cerca = difflib.get_close_matches(n.nombre, self.componentes, n=1)
            tip = f" ¿Quisiste decir '{cerca[0]}'?" if cerca else ""
            self.error(n.linea, f"el componente '{n.nombre}' no existe.{tip}")
            return []
        if len(n.args) != len(comp.params):
            self.error(n.linea, f"'{n.nombre}' espera {len(comp.params)} argumento(s) "
                                f"y recibió {len(n.args)}")
            return []
        if prof >= MAX_LLAMADAS:
            self.error(n.linea, f"recursión demasiado profunda en '{n.nombre}'")
            raise _Abortar()
        args = [self.valor(a, scope) for a in n.args]
        sc = dict(scope)
        sc.update(zip(comp.params, args))
        nuevo = (n.cuerpo, scope, ctx, prof) if n.cuerpo is not None else (None, None, None, prof)
        return self.expandir(comp.cuerpo, sc, nuevo, prof + 1)

    def elemento(self, n, scope, ctx, prof):
        self.cuenta += 1
        if self.cuenta > MAX_NODOS:
            self.error(n.linea, f"se superó el límite de {MAX_NODOS} elementos")
            raise _Abortar()
        tag, vacio = TAGS[n.tag]
        ids = [v for k, v in n.selectores if k == 'id']
        clases = [v for k, v in n.selectores if k == 'clase']
        otros = {}
        for k, v in n.attrs:
            val = self.valor(v, scope)
            if k == 'class':
                clases += val.split()
            elif k == 'id':
                ids.append(val)
            elif k in otros:
                self.error(n.linea, f"el atributo '{k}' está repetido en {n.tag}")
            else:
                otros[k] = val
        if len(ids) > 1:
            self.error(n.linea, f"{n.tag} tiene más de un id ({', '.join(ids)})")
        attrs = []
        if ids:
            attrs.append(('id', ids[0]))
        if clases:
            attrs.append(('class', ' '.join(clases)))
        attrs += list(otros.items())

        texto = None if n.texto is None else self.valor(n.texto, scope)
        if vacio and texto:                 # IMAGEN "foto.jpg" -> src ; INPUT "Nombre" -> placeholder
            clave = 'src' if tag == 'img' else 'placeholder'
            if clave in otros:
                self.aviso(n.linea, f"{n.tag}: el texto se ignora porque ya hay un atributo '{clave}'")
            else:
                attrs.append((clave, texto))
            texto = None

        hijos = []
        if n.cuerpo is not None:
            if vacio:
                self.error(n.linea, f"{n.tag} no admite bloque {{ }}")
            else:
                hijos = self.expandir(n.cuerpo, dict(scope), ctx, prof)
        return Nodo(tag, n.tag, attrs, texto, hijos, vacio, n.linea)

    def pagina(self, n, scope):
        for k, v in n.attrs:
            val = self.valor(v, scope)
            if k not in CLAVES_PAGINA:
                self.error(n.linea, f"PAGINA no tiene la opción '{k}' (válidas: {', '.join(CLAVES_PAGINA)})")
            elif k == 'tema' and val not in ('claro', 'oscuro', 'auto'):
                self.error(n.linea, "tema debe ser \"claro\", \"oscuro\" o \"auto\"")
            elif k == 'color' and not HEX.match(val):
                self.error(n.linea, f"color '{val}' no es un hexadecimal válido (ej. \"#0b7a75\")")
            else:
                self.config[k] = val

    # ---- validaciones sobre el árbol final ----
    def validar(self, nodos, ids, padre=None):
        for n in nodos:
            a = dict(n.attrs)
            if 'id' in a:
                if a['id'] in ids:
                    self.error(n.linea, f"el id '{a['id']}' está duplicado")
                ids.add(a['id'])
            if n.tag == 'img' and 'alt' not in a:
                self.aviso(n.linea, "IMAGEN sin atributo alt (accesibilidad)")
            if n.tag == 'a' and 'href' not in a:
                self.aviso(n.linea, "LINK sin href")
            if n.tag == 'li' and padre != 'ul':
                self.aviso(n.linea, "ITEM fuera de una LISTA")
            self.validar(n.hijos, ids, n.tag)

**Demostración semántica:** el mismo tipo de programa, ya expandido. `REPETIR 2` y el componente se transformaron en dos elementos reales, y `$i` y `$marca` se reemplazaron por sus valores.

In [ ]:
from tagscript_parser import parsear
from semantico import Analizador
from html import escape

def atributos(attrs):
    return "".join(f' {k}="{escape(v)}"' for k, v in attrs)

def mostrar_arbol(nodos, nivel=0):
    for n in nodos:
        txt = f'  "{n.texto}"' if n.texto else ""
        print(f"{'  ' * nivel}<{n.tag}{atributos(n.attrs)}>{txt}")
        mostrar_arbol(n.hijos, nivel + 1)

programa = '''VAR $marca = "TagScript"
COMPONENTE tarjeta($titulo) {
    DIV.card { ENCABEZADO $titulo }
}
REPETIR 2 { tarjeta("Número $i de $marca") }
'''
ast, _ = parsear(programa)
analizador = Analizador()
nodos, config = analizador.analizar(ast)

print(f"AST: {len(ast)} sentencias (VAR, COMPONENTE, REPETIR)")
print(f"Árbol resuelto: {len(nodos)} elementos de primer nivel\n")
mostrar_arbol(nodos)
print("\nDiagnósticos:", analizador.diagnosticos or "ninguno")

## 8. Generación de código

**HTML (`generador.py`).** Recorre el árbol resuelto y escribe una línea por etiqueta con sangría. Los textos y los valores de atributos se **escapan** con `html.escape` (un `<` o `&` del usuario no puede romper la página). Las etiquetas vacías (`img`, `input`, `hr`) se cierran con `/>`.

**CSS (`estilos.py`).** Se genera un `style.css` a partir de `PAGINA`:

- **Tema** claro, oscuro o automático (`prefers-color-scheme`).
- **Color de acento** a elección. El color del texto de los botones se calcula según el **contraste** (blanco o casi negro, el que resulte más legible).
- **Especificidad cero** con `:where()`: cualquier clase propia sobrescribe los estilos base.
- Clases listas para usar: `card`, `fila`, `grid`, `centrado`, `badge`, `btn-primary`, `secundario`.
- Diseño adaptable a celulares y respeto de `prefers-reduced-motion`.

In [ ]:
%%writefile build_tagscript/generador.py
"""Fase 4 - Generación de código: árbol de Nodo -> HTML."""
from html import escape

INDENT = "    "


def attrs_html(attrs):
    return "".join(f' {k}="{escape(v, quote=True)}"' for k, v in attrs)


def render(n, nivel):
    pad, a = INDENT * nivel, attrs_html(n.attrs)
    if n.vacio:
        return [f"{pad}<{n.tag}{a} />"]
    texto = escape(n.texto, quote=False) if n.texto else ""
    if not n.hijos:
        return [f"{pad}<{n.tag}{a}>{texto}</{n.tag}>"]
    out = [f"{pad}<{n.tag}{a}>"]
    if texto:
        out.append(f"{INDENT * (nivel + 1)}{texto}")
    for h in n.hijos:
        out += render(h, nivel + 1)
    out.append(f"{pad}</{n.tag}>")
    return out


def pagina_html(nodos, config, css_href="style.css"):
    cuerpo = []
    for n in nodos:
        cuerpo += render(n, 3)
    head = [
        '<meta charset="UTF-8">',
        '<meta name="viewport" content="width=device-width, initial-scale=1">',
        f'<title>{escape(config.get("titulo", "Página generada con TagScript"))}</title>',
    ]
    if 'descripcion' in config:
        head.append(f'<meta name="description" content="{escape(config["descripcion"], quote=True)}">')
    head.append(f'<link rel="stylesheet" href="{css_href}">')
    lineas = ["<!DOCTYPE html>", f'<html lang="{escape(config.get("idioma", "es"), quote=True)}">',
              f"{INDENT}<head>"]
    lineas += [INDENT * 2 + h for h in head]
    lineas += [f"{INDENT}</head>", f"{INDENT}<body>", f'{INDENT * 2}<main class="pagina">']
    lineas += cuerpo
    lineas += [f"{INDENT * 2}</main>", f"{INDENT}</body>", "</html>", ""]
    return "\n".join(lineas)

In [ ]:
%%writefile build_tagscript/estilos.py
"""Genera style.css a partir de la configuración de PAGINA.
Los estilos base usan :where() (especificidad 0), así que cualquier clase tuya los sobrescribe."""

COLOR_BASE = "#0b7a75"

CLARO = """--fondo:#f1f3f7; --superficie:#ffffff; --borde:#dde1ea;
    --tinta:#161a23; --suave:#566073;
    --enlace:color-mix(in srgb, var(--acento) 85%, #000);"""

OSCURO = """--fondo:#0e1016; --superficie:#171a23; --borde:#2a2f3d;
    --tinta:#eceff6; --suave:#9aa3b5;
    --enlace:color-mix(in srgb, var(--acento) 55%, #fff);"""


def _luminancia(hexa):
    h = hexa.lstrip('#')
    if len(h) == 3:
        h = ''.join(c * 2 for c in h)
    canales = [int(h[i:i + 2], 16) / 255 for i in (0, 2, 4)]
    f = lambda c: c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    r, g, b = map(f, canales)
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def _texto_sobre(hexa):
    """Blanco o casi negro, el que dé más contraste sobre el color de acento."""
    l = _luminancia(hexa)
    return '#ffffff' if 1.05 / (l + 0.05) >= (l + 0.05) / 0.05 else '#111111'


BASE = """/* Generado por TagScript. Tus clases sobrescriben estos estilos. */
:root {
    --acento: __ACENTO__;
    --sobre-acento: __SOBRE__;
    --serif: "Iowan Old Style", "Palatino Linotype", Palatino, "Source Serif 4", Georgia, serif;
    --sans: system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue", sans-serif;
    --mono: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
    __PALETA__
}

*, *::before, *::after { box-sizing: border-box; }

:where(body) {
    margin: 0;
    background: var(--fondo);
    color: var(--tinta);
    font: 400 1.0625rem/1.6 var(--sans);
    -webkit-font-smoothing: antialiased;
}

/* Apilado vertical con separación uniforme: no hace falta escribir márgenes */
:where(.pagina, div, section, header, footer, form, label) {
    display: flex;
    flex-direction: column;
    gap: 1rem;
}
:where(.pagina) {
    max-width: 44rem;
    margin: 0 auto;
    padding: clamp(1.5rem, 6vw, 4rem) clamp(1.25rem, 5vw, 2rem) 5rem;
    gap: 1.75rem;
}
:where(section) { gap: 1rem; }
:where(label) { gap: .4rem; font-size: .9rem; font-weight: 600; }

:where(h1) {
    margin: 0;
    font: 700 clamp(2.1rem, 6vw, 3.25rem)/1.05 var(--serif);
    letter-spacing: -.025em;
    text-wrap: balance;
}
:where(h2) {
    margin: .5rem 0 0;
    font: 600 1.5rem/1.2 var(--serif);
    letter-spacing: -.01em;
}
:where(h3) { margin: 0; font: 600 1.15rem/1.25 var(--serif); }
:where(p) { margin: 0; max-width: 62ch; color: var(--suave); }

:where(a) {
    color: var(--enlace);
    text-underline-offset: .2em;
    text-decoration-thickness: 1px;
}
:where(a):hover { text-decoration-thickness: 2px; }

/* Botones: <button> y cualquier LINK con clase btn-primary */
:where(button, a.btn-primary) {
    align-self: flex-start;
    display: inline-flex;
    align-items: center;
    justify-content: center;
    padding: .7em 1.25em;
    border: 0;
    border-radius: .625rem;
    background: var(--acento);
    color: var(--sobre-acento);
    font: 600 1rem/1.2 var(--sans);
    text-decoration: none;
    cursor: pointer;
    transition: background-color .15s;
}
:where(button, a.btn-primary):hover {
    background: color-mix(in srgb, var(--acento) 86%, #000);
}
:where(button, a.btn-primary):active { transform: translateY(1px); }
:where(button, a, input, textarea, select):focus-visible {
    outline: 3px solid color-mix(in srgb, var(--acento) 50%, transparent);
    outline-offset: 2px;
}
.secundario, .secundario:hover {
    background: transparent;
    color: var(--tinta);
    box-shadow: inset 0 0 0 1px var(--borde);
}
.secundario:hover { background: var(--superficie); }

:where(input, textarea, select) {
    width: 100%;
    padding: .7em .9em;
    border: 1px solid var(--borde);
    border-radius: .625rem;
    background: var(--superficie);
    color: var(--tinta);
    font: inherit;
    transition: border-color .15s, box-shadow .15s;
}
:where(input, textarea):focus { border-color: var(--acento); }
:where(input)::placeholder { color: var(--suave); opacity: .8; }

:where(nav) {
    display: flex;
    flex-direction: row;
    flex-wrap: wrap;
    align-items: center;
    gap: .5rem 1.5rem;
    padding-bottom: 1rem;
    border-bottom: 1px solid var(--borde);
}
:where(nav a) { color: var(--tinta); font-weight: 500; text-decoration: none; }
:where(nav a):hover { color: var(--enlace); }

:where(ul, ol) { margin: 0; padding-left: 1.25rem; }
:where(li + li) { margin-top: .35rem; }
:where(li)::marker { color: var(--acento); }

:where(img) { display: block; max-width: 100%; height: auto; border-radius: .75rem; }
:where(hr) { width: 100%; border: 0; border-top: 1px solid var(--borde); margin: .5rem 0; }
:where(footer) {
    padding-top: 1rem;
    border-top: 1px solid var(--borde);
    color: var(--suave);
    font-size: .9rem;
}
:where(code) {
    padding: .1em .4em;
    border: 1px solid var(--borde);
    border-radius: .35rem;
    background: var(--superficie);
    font: .9em var(--mono);
}

/* Clases listas para usar desde DIV.card, DIV.fila, DIV.grid, TEXTO.badge... */
.card {
    padding: clamp(1.25rem, 4vw, 1.75rem);
    border: 1px solid var(--borde);
    border-radius: 1rem;
    background: var(--superficie);
    gap: .6rem;
}
.fila { flex-direction: row; flex-wrap: wrap; align-items: center; gap: .75rem; }
.grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(11.5rem, 1fr)); gap: 1rem; }
.centrado { align-items: center; text-align: center; }
.badge {
    align-self: flex-start;
    padding: .2em .75em;
    border-radius: 99px;
    background: color-mix(in srgb, var(--acento) 14%, transparent);
    color: var(--enlace);
    font-size: .85rem;
    font-weight: 600;
}

@media (prefers-reduced-motion: reduce) {
    * { transition: none !important; }
}
"""


def generar_css(config):
    color = config.get('color', COLOR_BASE)
    tema = config.get('tema', 'auto')
    if tema == 'claro':
        paleta = CLARO + "\n    color-scheme: light;"
        extra = ""
    elif tema == 'oscuro':
        paleta = OSCURO + "\n    color-scheme: dark;"
        extra = ""
    else:
        paleta = CLARO + "\n    color-scheme: light dark;"
        extra = "\n@media (prefers-color-scheme: dark) {\n    :root {\n    " + OSCURO + "\n    }\n}\n"
    css = (BASE.replace('__ACENTO__', color)
               .replace('__SOBRE__', _texto_sobre(color))
               .replace('__PALETA__', paleta))
    return css.replace("\n*, *::before", extra + "\n*, *::before", 1)

### Compilador completo y utilidades

`compilar(texto)` ejecuta las cuatro fases y devuelve un `Resultado` con `html`, `css` y la lista de diagnósticos. Las utilidades de abajo sirven para **ver la página dentro de la notebook** (`vista_previa`), **guardar los archivos** (`guardar`) y **mostrar diagnósticos** con la línea de código que los provocó.

In [ ]:
%%writefile build_tagscript/compilador.py
"""Une las 4 fases: léxico -> sintaxis (AST) -> semántica -> generación."""
from nodos import Resultado
from tagscript_parser import parsear
from semantico import Analizador
from generador import pagina_html
from estilos import generar_css


def compilar(fuente):
    ast, diag = parsear(fuente)
    if ast is None:
        return Resultado(None, None, diag)
    an = Analizador()
    nodos, config = an.analizar(ast)
    diag = diag + an.diagnosticos
    if any(d.nivel == 'error' for d in diag):
        return Resultado(None, None, diag)
    return Resultado(pagina_html(nodos, config), generar_css(config), diag)

In [ ]:
%%writefile build_tagscript/ver.py
"""Muestra en la terminal lo que hace cada fase del compilador."""
from dataclasses import is_dataclass, fields
from lexer import lexer, errores_lexicos
from tagscript_parser import parsear
from semantico import Analizador
from generador import pagina_html, attrs_html
from estilos import generar_css
from nodos import Var, Nodo


def titulo(txt):
    print(f"\n{'=' * 64}\n  {txt}\n{'=' * 64}")


def obtener_tokens(fuente):
    errores_lexicos.clear()
    lexer.lineno = 1
    lexer.input(fuente)
    toks = []
    while True:
        t = lexer.token()
        if t is None:
            break
        toks.append(t)
    return toks


def fmt(v):
    if isinstance(v, Var):
        return "$" + v.nombre
    if isinstance(v, (list, tuple)):
        return "[" + ", ".join(fmt(x) for x in v) + "]"
    return repr(v)


def ver_ast(n, nivel=0):
    pad = "  " * nivel
    if isinstance(n, list):
        for x in n:
            ver_ast(x, nivel)
    elif is_dataclass(n):
        campos = [f"{f.name}={fmt(getattr(n, f.name))}" for f in fields(n)
                  if f.name not in ("cuerpo", "linea")]
        print(f"{pad}{type(n).__name__}  {'  '.join(campos)}")
        if getattr(n, "cuerpo", None):
            ver_ast(n.cuerpo, nivel + 1)


def ver_nodos(nodos, nivel=0):
    for n in nodos:
        txt = f'  "{n.texto}"' if n.texto else ""
        print(f"{'  ' * nivel}<{n.tag}{attrs_html(n.attrs)}>{txt}")
        ver_nodos(n.hijos, nivel + 1)


def contar(nodos):
    return sum(1 + contar(n.hijos) for n in nodos)


def mostrar_fases(fuente):
    titulo("FUENTE (lo que escribes)")
    for i, l in enumerate(fuente.splitlines(), 1):
        print(f"{i:>3} | {l}")

    titulo("FASE 1 - LÉXICO: el texto se corta en tokens")
    toks = obtener_tokens(fuente)
    print(f"{'línea':<6}{'token':<15}valor")
    for t in toks:
        print(f"{t.lineno:<6}{t.type:<15}{t.value!r}")
    print(f"\n{len(toks)} tokens. Comentarios, espacios y comas se descartan aquí.")

    titulo("FASE 2 - SINTÁCTICO: los tokens forman el árbol (AST)")
    ast, diag = parsear(fuente)
    if ast is None:
        print("No se pudo construir el árbol:")
        for d in diag:
            print(f"  línea {d.linea}: {d.mensaje}")
        return
    ver_ast(ast)
    print("\nAquí todavía hay COMPONENTE, REPETIR, VAR y $variables sin resolver.")

    titulo("FASE 3 - SEMÁNTICO: se expande y valida")
    an = Analizador()
    nodos, config = an.analizar(ast)
    ver_nodos(nodos)
    print(f"\nConfig de PAGINA: {config or '(por defecto)'}")
    print(f"{contar(nodos)} elementos reales. Ya no quedan componentes, bucles ni variables.")
    if an.diagnosticos:
        for d in an.diagnosticos:
            print(f"  {d.nivel} línea {d.linea}: {d.mensaje}")
    if any(d.nivel == "error" for d in an.diagnosticos):
        return

    titulo("FASE 4 - GENERACIÓN: HTML final")
    html = pagina_html(nodos, config)
    print(html)
    css = generar_css(config)
    n_fuente = len([l for l in fuente.splitlines() if l.strip() and not l.strip().startswith("//")])
    print(f"Resumen: {n_fuente} líneas de TagScript -> {len(html.splitlines())} de HTML "
          f"+ {len(css.splitlines())} de CSS")

In [ ]:
import html as _html
from pathlib import Path
from compilador import compilar
from ver import mostrar_fases

def pagina_autonoma(r):
    """Une HTML y CSS en un solo documento (para mostrarlo en la notebook)."""
    return r.html.replace('<link rel="stylesheet" href="style.css">',
                          f"<style>\n{r.css}\n</style>")

def vista_previa(r, alto=480):
    """Muestra la página generada dentro de la notebook."""
    try:
        from IPython.display import HTML, display
    except ImportError:
        print("(la vista previa solo se muestra dentro de Jupyter)")
        return
    doc = _html.escape(pagina_autonoma(r), quote=True)
    display(HTML(f'<iframe srcdoc="{doc}" style="width:100%;height:{alto}px;'
                 f'border:1px solid #ccc;border-radius:8px"></iframe>'))

def guardar(r, carpeta="salida"):
    """Escribe index.html y style.css en una carpeta."""
    destino = Path(carpeta)
    destino.mkdir(exist_ok=True)
    (destino / "index.html").write_text(r.html, encoding="utf-8")
    (destino / "style.css").write_text(r.css, encoding="utf-8")
    print(f"Archivos generados en {destino.resolve()}")

def mostrar_diagnosticos(r, fuente):
    lineas = fuente.splitlines()
    for d in sorted(r.diagnosticos, key=lambda d: d.linea or 0):
        donde = f"línea {d.linea}" if d.linea else "fin del archivo"
        print(f"  {d.nivel.upper():<6} ({donde}): {d.mensaje}")
        if d.linea and 1 <= d.linea <= len(lineas):
            print(f"         | {lineas[d.linea - 1].strip()}")

print("Compilador listo.")

## 9. Ejemplo simple

Un programa de cuatro líneas: una variable, un título con interpolación, un párrafo y un botón.

La celda siguiente muestra, una tras otra, las **cuatro fases** sobre este programa: tokens, AST, árbol resuelto y HTML final.

In [ ]:
EJEMPLO_SIMPLE = '''VAR $marca = "TagScript"
TITULO#principal "Bienvenido a $marca"
PARRAFO "Un traductor de TagScript a HTML y CSS."
BTN.btn-primary "Empezar"
'''

# Las cuatro fases, una tras otra
mostrar_fases(EJEMPLO_SIMPLE)

# La página resultante
resultado_simple = compilar(EJEMPLO_SIMPLE)
vista_previa(resultado_simple, alto=300)

**Acciones semánticas aplicadas en este ejemplo**

| Entrada | Acción | Resultado |
|---|---|---|
| `VAR $marca = "TagScript"` | Guarda `marca` en el ámbito | no genera HTML |
| `TITULO#principal` | `TITULO` → `h1`; el selector `#principal` pasa a atributo `id` | `<h1 id="principal">` |
| `"Bienvenido a $marca"` | Interpolación de la variable | `Bienvenido a TagScript` |
| `PARRAFO "..."` | `PARRAFO` → `p` | `<p>...</p>` |
| `BTN.btn-primary` | `BTN` → `button`; el selector `.btn-primary` pasa a atributo `class` | `<button class="btn-primary">` |
| (generación) | El CSS aplica estilos a `h1`, `p` y `.btn-primary` | `style.css` |

Los cuatro renglones de entrada producen un documento HTML completo más una hoja de estilos de más de 160 líneas.

## 10. Ejemplo complejo

Una página completa que usa todas las funciones del lenguaje:

- `PAGINA` para configurar título, tema y color.
- Una **variable** global (`$marca`) usada en varios lugares.
- Dos **componentes**: `funcion($nombre $detalle)` y `bloque($id $titulo)`. Este último usa `HIJOS` para recibir su contenido.
- Componentes **anidados** (las tarjetas dentro de un bloque) y reutilizados.
- Navegación, lista, formulario y pie de página.

La relación entre tamaño de entrada y de salida es la medida de qué tan eficiente es el lenguaje.

In [ ]:
EJEMPLO_COMPLEJO = '''// TagScript: una página completa en ~50 líneas
PAGINA [titulo="TagScript", tema="auto", color="#0b7a75", idioma="es",
        descripcion="Un lenguaje corto que compila a HTML y CSS"]

VAR $marca = "TagScript"

// Componentes: se definen una vez y se usan donde quieras
COMPONENTE funcion($nombre, $detalle) {
    DIV.card {
        ENCABEZADO $nombre
        PARRAFO $detalle
    }
}

COMPONENTE bloque($id, $titulo) {
    SECCION [id=$id] {
        SUBTITULO $titulo
        HIJOS                       // aquí entra lo que pongas entre { }
    }
}

NAV {
    LINK#inicio [href="#"] $marca
    LINK [href="#funciones"] "Funciones"
    LINK [href="#contacto"] "Contacto"
}

CABECERA {
    TEXTO.badge "Compilador DSL a HTML + CSS"
    TITULO#main-title "Escribe poco. Publica una página completa."
    PARRAFO "Cada línea de $marca se traduce a HTML limpio, con estilos listos y sin escribir etiquetas de cierre."
    DIV.fila {
        LINK.btn-primary [href="#funciones"] "Ver funciones"
        LINK [href="https://github.com"] "Repositorio"
    }
}

bloque("funciones", "Funciones") {
    DIV.grid {
        funcion("Componentes", "Defínelos una vez y reutilízalos con parámetros.")
        funcion("Variables", "Guarda un valor con VAR y úsalo con $nombre.")
        funcion("Bucles", "REPETIR genera listas y tablas sin copiar y pegar.")
    }
}

bloque("pasos", "Pasos del compilador") {
    LISTA {
        ITEM "Léxico: separa el texto en tokens"
        ITEM "Sintáctico: construye el árbol (AST)"
        ITEM "Semántico: expande componentes y valida"
        ITEM "Generación: escribe index.html y style.css"
    }
}

bloque("contacto", "Contacto") {
    FORM {
        CAMPO "Tu nombre" {
            INPUT#nombre [placeholder="Escribe tu nombre aquí..."]
        }
        DIV.fila {
            BTN#enviar "Enviar datos"
            BTN.secundario "Cancelar"
        }
    }
}

PIE { TEXTO "Hecho con $marca" }
'''

lineas_entrada = [l for l in EJEMPLO_COMPLEJO.splitlines() if l.strip() and not l.strip().startswith("//")]
resultado_complejo = compilar(EJEMPLO_COMPLEJO)

if resultado_complejo.ok:
    print(f"Entrada : {len(lineas_entrada)} líneas de TagScript")
    print(f"Salida  : {len(resultado_complejo.html.splitlines())} líneas de HTML "
          f"+ {len(resultado_complejo.css.splitlines())} líneas de CSS")
    print("\nPrimeras líneas del HTML generado:\n")
    print("\n".join(resultado_complejo.html.splitlines()[:24]))
    print("    ...")
    guardar(resultado_complejo, "salida")
else:
    mostrar_diagnosticos(resultado_complejo, EJEMPLO_COMPLEJO)

vista_previa(resultado_complejo, alto=720)

## 11. Manejo de errores

Un caso por fase. Cuando hay errores **no se genera ningún archivo**, para no dejar una página a medias.

| Fase | Qué se detecta |
|---|---|
| **Léxica** | Carácter no reconocido; comilla sin cerrar |
| **Sintáctica** | Falta una llave; atributo sin valor |
| **Semántica** | Variable no definida; componente inexistente (con sugerencia); argumentos de más o de menos; id repetido; bloque en una etiqueta vacía; recursión infinita; bucle gigante |
| **Avisos** | `IMAGEN` sin `alt`; `LINK` sin `href` |

Los dos últimos casos de la fase semántica muestran que el compilador **se protege solo** ante programas peligrosos.

In [ ]:
CASOS_ERROR = [
    ("LÉXICO", "Carácter que el lenguaje no conoce",
     'TITULO "Mi página"\nPARRAFO @ "Texto con un símbolo raro"'),
    ("LÉXICO", "Comilla sin cerrar",
     'TITULO "Mi página"\nPARRAFO "Este texto nunca cierra\nBTN "Aceptar"'),
    ("SINTÁCTICO", "Falta la llave que cierra el DIV",
     'DIV.card {\n    TITULO "Tarjeta"\n    PARRAFO "Falta la llave de cierre"'),
    ("SINTÁCTICO", "Atributo sin valor",
     'TITULO "Enlaces"\nLINK [href] "Ir al sitio"'),
    ("SEMÁNTICO", "Variable que nunca se definió",
     'VAR $marca = "TagScript"\nTITULO "Bienvenido a $marca"\nPARRAFO $nombre'),
    ("SEMÁNTICO", "Componente mal escrito (con sugerencia)",
     'COMPONENTE tarjeta($titulo) {\n    DIV.card { ENCABEZADO $titulo }\n}\n\ntarjta("Hola")'),
    ("SEMÁNTICO", "Cantidad de argumentos incorrecta",
     'COMPONENTE par($a $b) {\n    DIV { PARRAFO $a  PARRAFO $b }\n}\n\npar("uno")'),
    ("SEMÁNTICO", "Id repetido",
     'BTN#accion "Aceptar"\nBTN#accion "Cancelar"'),
    ("SEMÁNTICO", "Bloque { } dentro de una etiqueta vacía",
     'FORM {\n    INPUT [name="email"] {\n        PARRAFO "no cabe acá"\n    }\n}'),
    ("PROTECCIÓN", "Componente que se llama a sí mismo",
     'COMPONENTE bucle() {\n    bucle()\n}\n\nbucle()'),
    ("PROTECCIÓN", "REPETIR de un millón de elementos",
     'REPETIR 1000000 {\n    DIV "x"\n}'),
    ("AVISO", "Compila, pero advierte (accesibilidad)",
     'IMAGEN "foto.png"\nLINK "enlace sin destino"'),
]

for numero, (fase, titulo, codigo) in enumerate(CASOS_ERROR, 1):
    print(f"\n[{numero}] {fase}: {titulo}")
    for n, linea in enumerate(codigo.splitlines(), 1):
        print(f"    {n} | {linea}")
    r = compilar(codigo)
    mostrar_diagnosticos(r, codigo)
    print("    =>", "compiló" if r.ok else "NO compiló: no se genera ningún archivo")

## 12. Agente con Ollama

**Idea.** Un *agente* percibe, decide y actúa en un ciclo. Acá el ciclo es:

1. El usuario hace un **pedido en lenguaje natural** ("una página para una cafetería con tres platos").
2. Un modelo de lenguaje local, ejecutado con **Ollama**, escribe código **TagScript**.
3. El **compilador actúa como verificador**: si el código tiene errores, estos vuelven al modelo como retroalimentación.
4. El modelo corrige y se repite, hasta compilar o agotar el máximo de intentos.

**Por qué funciona bien:** los mensajes de error del compilador son precisos (fase, línea y sugerencia), así que el modelo recibe información concreta para corregirse y no necesita "adivinar".

**Para usarlo con un modelo real:**

1. Instalar Ollama desde ollama.com.
2. Descargar un modelo: `ollama pull llama3.2`.
3. Dejar el servicio corriendo (escucha en `http://localhost:11434`).

Para usar otro modelo se puede definir la variable de entorno `OLLAMA_MODEL`. El código detecta los modelos instalados y elige uno disponible.

**Modo simulado.** Si Ollama no está disponible, la demostración usa un modelo simulado que se equivoca a propósito y se corrige. Así el ciclo del agente puede mostrarse en cualquier computadora, sin depender de la red ni de la velocidad del modelo.

In [ ]:
import json, os, re, urllib.request
from dataclasses import dataclass, field
from typing import List, Optional

from nodos import TAGS
from compilador import compilar

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
if not OLLAMA_HOST.startswith("http"):
    OLLAMA_HOST = "http://" + OLLAMA_HOST
MODELO_PREFERIDO = os.environ.get("OLLAMA_MODEL", "llama3.2")

# ---------------------------------------------------------------- prompt
SISTEMA = """Eres un asistente que escribe código en TagScript, un lenguaje que se compila a HTML.
Reglas del lenguaje:
- Un elemento se escribe:  ETIQUETA#id.clase [atributo="valor"] "texto" { hijos }
  Todo lo que sigue a ETIQUETA es opcional. Solo se usan { } si hay hijos.
- Etiquetas válidas: __ETIQUETAS__
- IMAGEN, INPUT y LINEA no admiten { }. En IMAGEN "x" el texto es el src; en INPUT "x" es el placeholder.
- Comentarios con //
- Variables: VAR $nombre = "valor". Se usan como texto ($nombre) o dentro de un texto ("Hola $nombre").
- Componentes: COMPONENTE nombre($a $b) { ... } y se usan como nombre("x", "y").
  Dentro del componente, HIJOS inserta el bloque { } que se pasó en la llamada.
- Bucle: REPETIR 3 { ... }. Dentro se puede usar $i (1, 2, 3).
- Configuración opcional al inicio: PAGINA [titulo="...", tema="claro|oscuro|auto", color="#rrggbb"]
- Clases disponibles: card, fila, grid, centrado, badge, btn-primary, secundario.
Ejemplo:
PAGINA [titulo="Mi sitio", color="#0b7a75"]
TITULO "Hola"
DIV.grid {
    DIV.card { ENCABEZADO "Uno" PARRAFO "Primera tarjeta" }
    DIV.card { ENCABEZADO "Dos" PARRAFO "Segunda tarjeta" }
}
BTN.btn-primary "Contactar"
Responde SOLO con el código TagScript. Sin explicaciones y sin bloques de markdown.""".replace(
    "__ETIQUETAS__", ", ".join(TAGS))

# ---------------------------------------------------------------- Ollama
def ollama_modelos(timeout=2):
    """Lista de modelos instalados, o None si Ollama no responde."""
    try:
        with urllib.request.urlopen(f"{OLLAMA_HOST}/api/tags", timeout=timeout) as r:
            return [m["name"] for m in json.load(r).get("models", [])]
    except Exception:
        return None

def elegir_modelo(instalados):
    for nombre in instalados:
        if nombre.split(":")[0] == MODELO_PREFERIDO.split(":")[0]:
            return nombre
    return instalados[0] if instalados else None

def crear_llm_ollama(modelo, temperatura=0.2):
    """Devuelve una función llm(mensajes) -> texto que consulta a Ollama."""
    def llm(mensajes):
        cuerpo = json.dumps({"model": modelo, "messages": mensajes, "stream": False,
                             "options": {"temperature": temperatura}}).encode("utf-8")
        req = urllib.request.Request(f"{OLLAMA_HOST}/api/chat", data=cuerpo,
                                     headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=300) as r:
            return json.load(r)["message"]["content"]
    return llm

def crear_llm_simulado(respuestas):
    """Modelo falso para demostrar el ciclo sin Ollama: devuelve respuestas guionadas."""
    estado = {"i": 0}
    def llm(mensajes):
        k = min(estado["i"], len(respuestas) - 1)
        estado["i"] += 1
        return respuestas[k]
    return llm

# ---------------------------------------------------------------- agente
def limpiar_respuesta(texto):
    """Quita los bloques ``` que los modelos suelen agregar."""
    m = re.search(r"```[A-Za-z]*\n(.*?)```", texto, re.S)
    return (m.group(1) if m else texto).strip()

@dataclass
class Intento:
    numero: int
    codigo: str
    errores: list

@dataclass
class ResultadoAgente:
    ok: bool
    codigo: str
    resultado: Optional[object]
    intentos: List[Intento] = field(default_factory=list)

def informe_errores(errores, codigo):
    lineas = codigo.splitlines()
    partes = []
    for d in errores:
        texto = f"- línea {d.linea or '?'}: {d.mensaje}"
        if d.linea and 1 <= d.linea <= len(lineas):
            texto += f"   (código: {lineas[d.linea - 1].strip()})"
        partes.append(texto)
    return "\n".join(partes)

def agente(pedido, llm, max_intentos=3, verbose=True):
    """Ciclo: el LLM escribe -> el compilador verifica -> los errores vuelven al LLM."""
    mensajes = [{"role": "system", "content": SISTEMA},
                {"role": "user", "content": pedido}]
    intentos, codigo = [], ""
    for n in range(1, max_intentos + 1):
        codigo = limpiar_respuesta(llm(mensajes))
        r = compilar(codigo)
        errores = [d for d in r.diagnosticos if d.nivel == "error"]
        intentos.append(Intento(n, codigo, errores))
        if verbose:
            estado = "COMPILA" if r.ok else f"{len(errores)} error(es)"
            print(f"Intento {n}: {estado}")
            for d in errores:
                print(f"    línea {d.linea or '?'}: {d.mensaje}")
        if r.ok:
            return ResultadoAgente(True, codigo, r, intentos)
        mensajes.append({"role": "assistant", "content": codigo})
        mensajes.append({"role": "user", "content":
            "El compilador de TagScript rechazó tu código con estos errores:\n"
            + informe_errores(errores, codigo)
            + "\nCorrígelo y devuelve el código completo."})
    return ResultadoAgente(False, codigo, None, intentos)

print("Agente definido. Ollama disponible:", ollama_modelos() is not None)

In [ ]:
PEDIDO = ("Una página para la Cafetería Aurora: un título, tres platos del menú "
          "como tarjetas y un botón para reservar mesa.")

# Respuestas guionadas del modelo simulado: 1.º falta una llave, 2.º error de nombre, 3.º correcto
RESPUESTAS_SIMULADAS = [
'''PAGINA [titulo="Cafetería Aurora", color="#8a4b2a"]
TITULO "Cafetería Aurora"
DIV.grid {
    DIV.card { ENCABEZADO "Espresso" PARRAFO "Intenso y corto" }
''',
'''PAGINA [titulo="Cafetería Aurora", color="#8a4b2a"]
COMPONENTE plato($nombre $detalle) {
    DIV.card { ENCABEZADO $nombre  PARRAFO $detalle }
}
TITULO "Cafetería Aurora"
DIV.grid {
    plto("Espresso", "Intenso y corto")
    plato("Medialuna", "Con manteca")
}
''',
'''PAGINA [titulo="Cafetería Aurora", color="#8a4b2a"]
COMPONENTE plato($nombre $detalle) {
    DIV.card { ENCABEZADO $nombre  PARRAFO $detalle }
}
TITULO "Cafetería Aurora"
DIV.grid {
    plato("Espresso", "Intenso y corto")
    plato("Medialuna", "Con manteca")
    plato("Licuado", "De frutas de estación")
}
BTN.btn-primary "Reservar mesa"
''']

instalados = ollama_modelos()
if instalados:
    modelo = elegir_modelo(instalados)
    print(f"Usando Ollama con el modelo '{modelo}'\n")
    llm = crear_llm_ollama(modelo)
else:
    print("Ollama no responde: se usa el MODELO SIMULADO (el ciclo es el mismo).\n")
    llm = crear_llm_simulado(RESPUESTAS_SIMULADAS)

print("PEDIDO:", PEDIDO, "\n")
resultado_agente = agente(PEDIDO, llm, max_intentos=4)

if resultado_agente.ok:
    print("\nCÓDIGO FINAL EN TAGSCRIPT:\n")
    print(resultado_agente.codigo)
    vista_previa(resultado_agente.resultado, alto=520)
else:
    print("\nEl agente no logró un código válido. Último intento:\n")
    print(resultado_agente.codigo)

## Conclusiones

**Lo logrado**

- Un traductor completo con las cuatro fases bien separadas: léxica, sintáctica, semántica y generación.
- Un lenguaje con **abstracciones reales** (componentes, variables y bucles), que reduce el texto de entrada frente al HTML equivalente.
- Errores **precisos** (fase, línea y sugerencias) y un compilador que **se protege** de entradas peligrosas.
- Un **agente** que cierra el ciclo con un modelo de lenguaje local usando al compilador como verificador.

**Limitaciones**

- No hay condicionales ni expresiones; los estilos se limitan a clases predefinidas.
- Un error sintáctico se informa de a uno (se detiene en el primero).
- La calidad del agente depende del modelo instalado; modelos pequeños pueden necesitar más intentos.